# Normalizacao de Dados de Produtos - Questao 2

## Objetivo
Realizar a normalizacao do dataset `produtos_raw.csv`, tratando inconsistencias nas categorias, convertendo precos para formato numerico e removendo registros duplicados.

## Etapas
1. Padronizacao das categorias (39 variacoes mapeadas para 3 categorias padrão)
2. Conversao da coluna de precos de texto para numerico
3. Remocao de duplicatas pela coluna `code`

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import warnings

warnings.filterwarnings('ignore')

## 1. Carregamento dos Dados

In [2]:
df = pd.read_csv('../datasets/produtos_raw.csv')

print(f"Dataset carregado: {df.shape[0]} linhas, {df.shape[1]} colunas")
print(f"Colunas: {list(df.columns)}")
print(f"\nPrimeiras linhas:")
df.head(10)

Dataset carregado: 157 linhas, 4 colunas
Colunas: ['name', 'price', 'code', 'actual_category']

Primeiras linhas:


,name,price,code,actual_category
0,Transponder AIS Maré Magnum,R$ 33122.52,1,ELETRONICOS
1,Transponder Furuno Marlin,R$ 13998.15,2,ELETRONICOS
2,Radar Furuno Pulse Leviathan,R$ 9024.19,3,E L E T R Ô N I C O S
3,Rádio AIS Hydro Tidal Zen,R$ 3381.88,4,Eletrunicos
4,Piloto Automático Furuno Storm,R$ 23669.01,5,Eletronicoz
5,Transponder AIS Vector,R$ 11820.21,6,Eletrunicos
6,Radar AIS Zen,R$ 19518.77,7,eLeTrÔnIcOs
7,GPS AIS Zen,R$ 4984.15,8,E L E T R Ô N I C O S
8,Transponder AIS Titan Pulse,R$ 39705.5,9,Eletronicoz
9,Piloto Automático Simrad Titan Flux Magnum,R$ 32033.04,10,eletrônicos


In [3]:
print("Tipos de dados:")
print(df.dtypes)
print(f"\nValores nulos por coluna:")
print(df.isnull().sum())
print(f"\nCodigos unicos: {df['code'].nunique()} de {len(df)} registros")

Tipos de dados:
name               object
price              object
code                int64
actual_category    object
dtype: object

Valores nulos por coluna:
name               0
price              0
code               0
actual_category    0
dtype: int64

Codigos unicos: 150 de 157 registros


## 2. Parte 1 - Padronizacao de Categorias

A coluna `actual_category` apresenta 39 variacoes de escrita para apenas 3 categorias reais:
- **eletronicos** (variacoes com acentuacao, espacamento, erros ortograficos)
- **propulsao** (variacoes similares)
- **ancoragem** (variacoes similares)

Estrategia: remover espacos, converter para minusculo, remover acentos e aplicar correspondencia por substring.

In [4]:
print(f"Quantidade de categorias distintas (antes): {df['actual_category'].nunique()}")
print(f"\nTodas as variacoes encontradas:")
print(df['actual_category'].value_counts().to_string())

Quantidade de categorias distintas (antes): 39

Todas as variacoes encontradas:
actual_category
AncorageM                9
Propução                 8
Ancoraguem               8
eletrônicos              7
Eletronicoz              7
E L E T R Ô N I C O S    6
ELETRONICOS              6
P R O P U L S Ã O        6
propulsão                6
PROPULSAO                6
Eletrunicos              5
Ancorajm                 5
eLeTrÔnIcOs              5
Propulssão               5
Propulção                5
Prop                     5
Encoragem                5
aNcOrAgEm                5
A N C O R A G E M        5
Eletrônicos              4
propulsao                4
EletrônicoS              3
Propulçao                3
Ancorajem                3
eletronicos              3
Ancoragem                3
Ancorajen                2
Eletronicos              2
Propulsam                2
Eletroniscos             2
pRoPuLsÃo                2
ancoragem                2
AnCoRaGeM                2
ELEtRÔNICOS  

In [5]:
def remover_acentos(texto):
    """Remove acentos e diacriticos de uma string."""
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))


def normalizar_categoria(valor):
    """Normaliza a categoria removendo espacos, acentos e aplicando correspondencia por substring."""
    texto = remover_acentos(str(valor).replace(' ', '').lower())
    
    if 'eletr' in texto:
        return 'eletrônicos'
    elif 'prop' in texto:
        return 'propulsão'
    elif 'ancor' in texto or 'encor' in texto:
        return 'ancoragem'
    else:
        return 'NAO_CLASSIFICADO'


df['category'] = df['actual_category'].apply(normalizar_categoria)

nao_classificados = (df['category'] == 'NAO_CLASSIFICADO').sum()
print(f"Registros nao classificados: {nao_classificados}")

Registros nao classificados: 0


In [6]:
print(f"Quantidade de categorias distintas (apos normalizacao): {df['category'].nunique()}")
print(f"\nDistribuicao das categorias padronizadas:")
print(df['category'].value_counts())

Quantidade de categorias distintas (apos normalizacao): 3

Distribuicao das categorias padronizadas:
category
propulsão      53
ancoragem      53
eletrônicos    51
Name: count, dtype: int64


## 3. Parte 2 - Conversao de Precos

A coluna `price` esta em formato texto com o prefixo `R$ `. Necessario converter para tipo numerico (float).

In [7]:
print("Formato original da coluna 'price':")
print(df['price'].head(10))
print(f"\nTipo atual: {df['price'].dtype}")

Formato original da coluna 'price':
0    R$ 33122.52
1    R$ 13998.15
2     R$ 9024.19
3     R$ 3381.88
4    R$ 23669.01
5    R$ 11820.21
6    R$ 19518.77
7     R$ 4984.15
8     R$ 39705.5
9    R$ 32033.04
Name: price, dtype: object

Tipo atual: object


In [8]:
df['price'] = df['price'].str.replace('R$ ', '', regex=False).astype(float)

print(f"Tipo apos conversao: {df['price'].dtype}")
print(f"\nEstatisticas de preco:")
print(df['price'].describe())

Tipo apos conversao: float64

Estatisticas de preco:
count       157.000000
mean      35194.622484
std       42183.183409
min         309.540000
25%        3769.930000
50%       13704.100000
75%       51634.040000
max      148198.230000
Name: price, dtype: float64


## 4. Parte 3 - Remocao de Duplicatas

O dataset possui codigos de produto duplicados. Serao mantidos apenas os primeiros registros de cada codigo.

In [9]:
total_antes = len(df)
duplicatas = df[df.duplicated(subset='code', keep='first')]

print(f"Total de registros antes: {total_antes}")
print(f"Codigos unicos: {df['code'].nunique()}")
print(f"Duplicatas identificadas: {len(duplicatas)}")
print(f"\nCodigos duplicados:")
print(df[df.duplicated(subset='code', keep=False)].sort_values('code')[['code', 'name', 'price', 'category']])

Total de registros antes: 157
Codigos unicos: 150
Duplicatas identificadas: 7

Codigos duplicados:
     code                                    name      price     category
36     37            GPS Lowrance Evo Storm Drift    6067.71  eletrônicos
37     37            GPS Lowrance Evo Storm Drift    6067.71  eletrônicos
62     62       Motor Diesel Yanmar Velocity 37HP  102221.97    propulsão
63     62       Motor Diesel Yanmar Velocity 37HP  102221.97    propulsão
64     62       Motor Diesel Yanmar Velocity 37HP  102221.97    propulsão
65     62       Motor Diesel Yanmar Velocity 37HP  102221.97    propulsão
131   127  Cabo de Nylon Delta Velocity Core Mako    1549.35    ancoragem
132   127  Cabo de Nylon Delta Velocity Core Mako    1549.35    ancoragem
124   145         Boia de Arqueamento Delta Nexus    4349.86    ancoragem
150   145         Boia de Arqueamento Delta Nexus    4349.86    ancoragem
151   145         Boia de Arqueamento Delta Nexus    4349.86    ancoragem


In [10]:
df = df.drop_duplicates(subset='code', keep='first').reset_index(drop=True)

total_depois = len(df)
duplicatas_removidas = total_antes - total_depois

print(f"Total de registros apos remocao: {total_depois}")
print(f"Duplicatas removidas: {duplicatas_removidas}")

Total de registros apos remocao: 150
Duplicatas removidas: 7


## 5. Resultado Final

In [11]:
df_final = df[['name', 'price', 'code', 'category']].copy()

print("=" * 60)
print("RESUMO DO PROCESSAMENTO")
print("=" * 60)
print(f"Registros originais: {total_antes}")
print(f"Registros finais: {len(df_final)}")
print(f"Duplicatas removidas: {duplicatas_removidas}")
print(f"Categorias padronizadas: {df_final['category'].nunique()}")
print(f"Tipo da coluna price: {df_final['price'].dtype}")
print(f"\nDistribuicao por categoria:")
print(df_final['category'].value_counts())
print(f"\nAmostra do dataset final:")
df_final.head(10)

RESUMO DO PROCESSAMENTO
Registros originais: 157
Registros finais: 150
Duplicatas removidas: 7
Categorias padronizadas: 3
Tipo da coluna price: float64

Distribuicao por categoria:
category
eletrônicos    50
propulsão      50
ancoragem      50
Name: count, dtype: int64

Amostra do dataset final:


,name,price,code,category
0,Transponder AIS Maré Magnum,33122.52,1,eletrônicos
1,Transponder Furuno Marlin,13998.15,2,eletrônicos
2,Radar Furuno Pulse Leviathan,9024.19,3,eletrônicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,eletrônicos
4,Piloto Automático Furuno Storm,23669.01,5,eletrônicos
5,Transponder AIS Vector,11820.21,6,eletrônicos
6,Radar AIS Zen,19518.77,7,eletrônicos
7,GPS AIS Zen,4984.15,8,eletrônicos
8,Transponder AIS Titan Pulse,39705.50,9,eletrônicos
9,Piloto Automático Simrad Titan Flux Magnum,32033.04,10,eletrônicos


In [12]:
print("Tipos de dados finais:")
print(df_final.dtypes)
print(f"\nInfo:")
df_final.info()

Tipos de dados finais:
name         object
price       float64
code          int64
category     object
dtype: object

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      150 non-null    object 
 1   price     150 non-null    float64
 2   code      150 non-null    int64  
 3   category  150 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 4.8+ KB


## 6. Exportacao dos Dados Processados

In [13]:
df_final.to_csv('../outputs/dados_processados/produtos_normalizados.csv', index=False)
print("Arquivo salvo em: ../outputs/dados_processados/produtos_normalizados.csv")
print(f"Registros exportados: {len(df_final)}")

Arquivo salvo em: ../outputs/dados_processados/produtos_normalizados.csv
Registros exportados: 150


## 7. Respostas as Validacoes

In [14]:
print("=" * 60)
print("RESPOSTAS - QUESTAO 2")
print("=" * 60)

print(f"\nQuestao 2.2 - Quantas duplicatas foram removidas?")
print(f"RESPOSTA: {duplicatas_removidas} duplicatas removidas ({total_antes} - {total_depois} = {duplicatas_removidas})")

RESPOSTAS - QUESTAO 2

Questao 2.2 - Quantas duplicatas foram removidas?
RESPOSTA: 7 duplicatas removidas (157 - 150 = 7)


## SQL de Referencia (Questao 2.1)

Equivalente SQL para o processo de normalizacao realizado neste notebook.

```sql
-- Etapa 1: Padronizacao de categorias
-- Utiliza CASE com LIKE para mapear variacoes para categorias padrão

WITH categorias_normalizadas AS (
    SELECT
        name,
        CAST(REPLACE(price, 'R$ ', '') AS DECIMAL(10, 2)) AS price,
        code,
        CASE
            WHEN LOWER(REPLACE(actual_category, ' ', '')) LIKE '%eletr%'
                THEN 'eletrônicos'
            WHEN LOWER(REPLACE(actual_category, ' ', '')) LIKE '%prop%'
                THEN 'propulsão'
            WHEN LOWER(REPLACE(actual_category, ' ', '')) LIKE '%ancor%'
                OR LOWER(REPLACE(actual_category, ' ', '')) LIKE '%encor%'
                THEN 'ancoragem'
            ELSE 'NAO_CLASSIFICADO'
        END AS category
    FROM produtos_raw
),

-- Etapa 2: Remocao de duplicatas (manter primeiro registro por code)
produtos_ranqueados AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY code ORDER BY code) AS rn
    FROM categorias_normalizadas
)

SELECT
    name,
    price,
    code,
    category
FROM produtos_ranqueados
WHERE rn = 1
ORDER BY code;
```